<a href="https://colab.research.google.com/github/Bhavesh43-OG/bhaveshd/blob/main/Bhavesh_Desale_Exp10_ANOVA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

titanic_path = kagglehub.competition_download('titanic')

print('Data source import complete.')


# Python for Data 26: ANOVA
[back to index](https://www.kaggle.com/hamelg/python-for-data-analysis-index)

In lesson 24 we introduced the t-test for checking whether the means of two groups differ. The t-test works well when dealing with two groups, but sometimes we want to compare more than two groups at the same time. For example, if we wanted to test whether voter age differs based on some categorical variable like race, we have to compare the means of each level or group the variable. We could carry out a separate t-test for each pair of groups, but when you conduct many tests you increase the chances of false positives. The [analysis of variance](https://en.wikipedia.org/wiki/Analysis_of_variance) or ANOVA is a statistical inference test that lets you compare multiple groups at the same time.

# One-Way ANOVA

The one-way ANOVA tests whether the mean of some numeric variable differs across the levels of one categorical variable. It essentially answers the question: do any of the group means differ from one another? We won't get into the details of carrying out an ANOVA by hand as it involves more calculations than the t-test, but the process is similar: you go through several calculations to arrive at a test statistic and then you compare the test statistic to a critical value based on a probability distribution. In the case of the ANOVA, you use the "[f-distribution](https://en.wikipedia.org/wiki/F-distribution)".

The scipy library has a function for carrying out one-way ANOVA tests called scipy.stats.f_oneway(). Let's generate some fake voter age and demographic data and use the ANOVA to compare average ages across the groups:

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as stats

In [ ]:
np.random.seed(12)

races =   ["asian","black","hispanic","other","white"]

# Generate random data
voter_race = np.random.choice(a= races,
                              p = [0.05, 0.15 ,0.25, 0.05, 0.5],
                              size=1000)

voter_age = stats.poisson.rvs(loc=18,
                              mu=30,
                              size=1000)

# Group age data by race
voter_frame = pd.DataFrame({"race":voter_race,"age":voter_age})
groups = voter_frame.groupby("race").groups

# Etract individual groups
asian = voter_age[groups["asian"]]
black = voter_age[groups["black"]]
hispanic = voter_age[groups["hispanic"]]
other = voter_age[groups["other"]]
white = voter_age[groups["white"]]

# Perform the ANOVA
stats.f_oneway(asian, black, hispanic, other, white)

The test output yields an F-statistic of 1.774 and a p-value of 0.1317, indicating that there is no significant difference between the means of each group.

Another way to carry out an ANOVA test is to use the statsmodels library, which allows you to specify a model with a formula syntax that mirrors that used by the R programming language. R users may find this method more familiar:

In [ ]:
import statsmodels.api as sm
from statsmodels.formula.api import ols

model = ols('age ~ race',                 # Model formula
            data = voter_frame).fit()

anova_result = sm.stats.anova_lm(model, typ=2)
print (anova_result)

As you can see, the statsmodels method produced the same F statistic and P-value (listed as PR(<F)) as the stats.f_oneway method.

Now let's make new age data where the group means do differ and run a second ANOVA:

In [ ]:
np.random.seed(12)

# Generate random data
voter_race = np.random.choice(a= races,
                              p = [0.05, 0.15 ,0.25, 0.05, 0.5],
                              size=1000)

# Use a different distribution for white ages
white_ages = stats.poisson.rvs(loc=18,
                              mu=32,
                              size=1000)

voter_age = stats.poisson.rvs(loc=18,
                              mu=30,
                              size=1000)

voter_age = np.where(voter_race=="white", white_ages, voter_age)

# Group age data by race
voter_frame = pd.DataFrame({"race":voter_race,"age":voter_age})
groups = voter_frame.groupby("race").groups

# Extract individual groups
asian = voter_age[groups["asian"]]
black = voter_age[groups["black"]]
hispanic = voter_age[groups["hispanic"]]
other = voter_age[groups["other"]]
white = voter_age[groups["white"]]

# Perform the ANOVA
stats.f_oneway(asian, black, hispanic, other, white)

In [ ]:
# Alternate method
model = ols('age ~ race',                 # Model formula
            data = voter_frame).fit()

anova_result = sm.stats.anova_lm(model, typ=2)
print (anova_result)

The test result suggests the groups don't have the same sample means in this case, since the p-value is significant at a 99% confidence level. We know that it is the white voters who differ because we set it up that way in the code, but when testing real data, you may not know which group(s) caused the test to throw a positive result. To check which groups differ after getting a positive ANOVA result, you can perform a follow up test or "post-hoc test".

One post-hoc test is to perform a separate t-test for each pair of groups. You can perform a t-test between all pairs using by running each pair through the stats.ttest_ind() we covered in the lesson on t-tests:

In [ ]:
# Get all race pairs
race_pairs = []

for race1 in range(4):
    for race2  in range(race1+1,5):
        race_pairs.append((races[race1], races[race2]))

# Conduct t-test on each pair
for race1, race2 in race_pairs:
    print(race1, race2)
    print(stats.ttest_ind(voter_age[groups[race1]],
                          voter_age[groups[race2]]))

The p-values for each pairwise t-test suggest mean of white voters is likely different from the other groups, since the p-values for each t-test involving the white group is below 0.05. Using unadjusted pairwise t-tests can overestimate significance, however, because the more comparisons you make, the more likely you are to come across an unlikely result due to chance. We can adjust for this multiple comparison problem by dividing the statistical significance level by the number of comparisons made. In this case, if we were looking for a significance level of 5%, we'd be looking for p-values of 0.05/10 = 0.005 or less. This simple adjustment for multiple comparisons is known as the Bonferroni correction.

The Bonferroni correction is a conservative approach to account for the multiple comparisons problem that may end up rejecting results that are actually significant. Another common post hoc-test is Tukey's test. You can carry out Tukey's test using the pairwise_tukeyhsd() function in the statsmodels.stats.multicomp library:

In [ ]:
from statsmodels.stats.multicomp import pairwise_tukeyhsd

tukey = pairwise_tukeyhsd(endog=voter_age,     # Data
                          groups=voter_race,   # Groups
                          alpha=0.05)          # Significance level

tukey.plot_simultaneous()    # Plot group confidence intervals
plt.vlines(x=49.57,ymin=-0.5,ymax=4.5, color="red")

tukey.summary()              # See test summary

The output of the Tukey test shows the average difference, a confidence interval as well as whether you should reject the null hypothesis for each pair of groups at the given significance level. In this case, the test suggests we reject the null hypothesis for 3 pairs, with each pair including the "white" category. This suggests the white group is likely different from the others. The 95% confidence interval plot reinforces the results visually: only 1 other group's confidence interval overlaps the white group's confidence interval.

Q1. Explain input and output characteristics for f_oneway function from stats library

Ans. The function **f_oneway** (from `scipy.stats`) is used to perform a **One-Way ANOVA (Analysis of Variance)** test. It checks whether the **means of three or more independent groups are significantly different**.


Input Characteristics

1. Multiple Independent Samples

* Accepts **two or more groups** (arrays/lists):

  ```python
  f_oneway(group1, group2, group3, ...)
  ```
Important:

* Each group represents a **different category**
* Groups must be **independent** (not paired)


2. Data Type

* Numerical data (**continuous values**)
* Example: marks, height, income, etc.


3. Assumptions

* Samples are **independent**
* Data in each group is **normally distributed**
* Variance across groups is **approximately equal** (homogeneity)



4. Equal or Unequal Size

* Groups can have:

  * Equal size ✅
  * Unequal size ✅



Output Characteristics

Returns Two Values

1. F-statistic

* Ratio of:

  * Variance **between groups**
  * Variance **within groups**

 Larger F-value → greater difference between group means



2. p-value

* Indicates statistical significance

 Decision:

* If **p-value < α (0.05)**
  ➝ Reject H₀
  ➝ At least one group mean is different

* Else
  ➝ Accept H₀
  ➝ All group means are similar



 Hypotheses

* **H₀:** All group means are equal
* **H₁:** At least one mean is different



 Example

```python
from scipy.stats import f_oneway

g1 = [10, 12, 14]
g2 = [20, 22, 24]
g3 = [30, 32, 34]

f_stat, p_val = f_oneway(g1, g2, g3)
```



Q2. Explain results of sm.stats.anova_lm fucntion.

Ans. The function **sm.stats.anova_lm** (from `statsmodels`) is used to generate an **ANOVA table** for a fitted linear model. It helps determine whether **independent variables significantly affect the dependent variable**.


Output of `sm.stats.anova_lm()`

The function returns an **ANOVA table** (in tabular form) with the following columns:


 1. sum_sq (Sum of Squares)

* Measures **variation** in the data

Types:

* **Between groups (model)** → variation explained by model
* **Residual (error)** → unexplained variation

 Higher model sum_sq → model explains more variability


 2. df (Degrees of Freedom)

* Number of independent values used in calculation

 Helps compute mean squares



 3. mean_sq (Mean Square)

* Calculated as:

[
\text{mean_sq} = \frac{\text{sum_sq}}{\text{df}}
]

Represents average variation


4. F (F-statistic)

* Ratio of:

  * Model variance
  * Error variance
Larger F → stronger evidence that groups differ


5. PR(>F) (p-value)

* Probability of observing F-statistic under H₀

 Decision:

* If **p-value < α (0.05)**
  ➝ Reject H₀
  ➝ Variable has **significant effect**

* Else
  ➝ No significant effect


 Interpretation of Results

Example conclusion:

* If p-value is small → independent variable significantly affects output
* If p-value is large → no significant impact



Q3. What role ANOVA plays in data analytics?

Ans. **ANOVA (Analysis of Variance)** plays a very important role in data analytics by helping us **compare multiple groups and make data-driven decisions**.



Role of ANOVA in Data Analytics

1. Comparing Multiple Group Means

 ANOVA helps check whether **three or more groups have different means**

**Example:**

* Sales performance in different regions

* Marks of students from different classes

 Instead of multiple t-tests, ANOVA gives a **single, reliable test**



2. Identifying Significant Factors

 Helps find which **independent variables (factors)** affect the outcome

**Example:**

* Does advertising type affect sales?
* Does fertilizer type affect crop yield?

Useful in **feature selection**



3. Reducing Error in Analysis

* Performing multiple t-tests increases **Type I error**
* ANOVA controls this error and gives **accurate results**


4. Supporting Model Building

 Used in:

* Regression analysis
* Experimental design

 Helps validate whether a model or factor is **statistically significant**



 5. Business Decision Making

 Helps organizations:

* Compare strategies
* Optimize processes
* Improve performance

**Example:**

* Which marketing campaign works best?
* Which product variation is preferred?



6. Basis for Advanced Analysis

* Leads to **post-hoc tests** (Tukey, Bonferroni)
* Used in **machine learning preprocessing**


Real-Life Applications

* Healthcare → comparing treatments
* Marketing → comparing campaigns
* Manufacturing → quality control
* Education → comparing teaching methods


Q4. When to use one way ANOVA? Explain with real world case.

Ans. We use **One-way ANOVA** when we want to compare the **means of three or more independent groups based on one factor (independent variable)**.


 When to Use One-Way ANOVA

 Use it when:

 1. One Independent Variable (Factor)

* Only **one factor** affects the outcome
* This factor has **3 or more categories (groups)**

 Example:

* Teaching method (Online, Offline, Hybrid)

2. One Dependent Variable

* A **numerical (continuous)** variable

 Example:

* Marks, income, height



 3. Independent Groups

* Observations in each group are **independent**
* No pairing (unlike paired t-test)



 4. Assumptions

* Data is approximately **normally distributed**
* Variances are **equal (homogeneity)**



 Real-World Case

 Example: Comparing Student Performance
 Problem:
A college wants to check whether **teaching method affects student marks**

Groups (Independent Variable):

* Method 1: Online
* Method 2: Offline
* Method 3: Hybrid

 Dependent Variable:

* Student marks (numerical)



Hypothesis

* **H₀:** All group means are equal
* **H₁:** At least one group mean is different


Interpretation

* If **p-value < 0.05**
  ➝ Reject H₀
  ➝ Teaching method has **significant effect**

* If **p-value ≥ 0.05**
  ➝ No significant difference



Q5.  When to use 2 way ANOVA ? Explain with real world case.

Ans. **Two-way ANOVA (Analysis of Variance)** is used when you want to study the effect of **two independent variables (factors)** on **one dependent variable**, and also check whether there is an **interaction effect** between those two factors.


**When to use Two-Way ANOVA**

Use it when:

1. You have **two categorical independent variables (factors)**
2. You have **one continuous dependent variable**
3. You want to analyze:

   * The effect of **Factor A**
   * The effect of **Factor B**
   * The **interaction effect (A × B)**

 **Key Purpose**

* To understand **individual effects** of two factors
* To determine whether the **combined effect of both factors is different from their individual effects**



**Real-World Example**

**Case: Student Performance Analysis**

Suppose a college wants to analyze student marks based on:

* **Factor 1:** Teaching Method

  * Online
  * Offline

* **Factor 2:** Study Time

  * Less than 2 hours
  * More than 2 hours

* **Dependent Variable:** Exam Marks


**What Two-Way ANOVA Tests Here**

1. **Effect of Teaching Method**

   * Do students perform differently in online vs offline classes?

2. **Effect of Study Time**

   * Does study time impact marks?

3. **Interaction Effect (Important!)**

   * Does the effectiveness of teaching method **depend on study time**?

Example insight:

* Online teaching may work well **only when students study more**
* Offline teaching may perform better **even with less study time**

